## XGBoost

O [XGBoost](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html) (Extreme Gradient Boosting) é um modelo baseado em árvores que utiliza a técnica de boosting, onde múltiplos modelos fracos são combinados sequencialmente para formar um modelo forte.

A cada nova árvore, o modelo tenta corrigir os erros das árvores anteriores, melhorando progressivamente o desempenho.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores no modelo
  - Mais árvores → melhor aprendizado (até certo limite)

- **learning_rate**: taxa de aprendizado
  - Valores menores tornam o aprendizado mais lento, porém mais robusto

- **max_depth**: profundidade das árvores
  - Controla a complexidade do modelo

Estratégia

Foi utilizado GridSearchCV com validação cruzada (5 folds) e métrica ROC AUC para encontrar a melhor combinação de hiperparâmetros.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from xgboost import XGBClassifier
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
    from sklearn.model_selection import GridSearchCV
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []

# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote,
#         )

#         scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

#         steps = []

#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('xgb', XGBClassifier(random_state=42)))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1]

#         results.append({
#             "model": "XGBoost",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "timestamp": pd.Timestamp.now()
#         })


# df_result = pd.DataFrame(results)

# save_results(df_result)

# display(df_result)


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )
    print(f"Colunas utilizadas para o cenario '{scenario}': {X_train.columns.tolist()}")
    print(f"Total de features: {len(X_train.columns)}")

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    param_grid_xgb = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "xgb__n_estimators": [200, 400],
        "xgb__learning_rate": [0.05, 0.1],
        "xgb__max_depth": [3, 5],
        "xgb__subsample": [0.8, 1],
        "xgb__colsample_bytree": [0.8, 1]
    }

    grid_xgb = GridSearchCV(
        estimator=Pipeline([
            ('smote', SMOTE(random_state=42)),
            ('xgb', XGBClassifier(eval_metric="logloss",
            random_state=42,
            scale_pos_weight=scale_pos_weight,))
        ]),
        param_grid=param_grid_xgb,
        cv=5,
        scoring="roc_auc",
        n_jobs=2
    )

    grid_xgb.fit(X_train, y_train)

    print("Best parameters:", grid_xgb.best_params_)
    print("Best ROC AUC (CV):", grid_xgb.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_xgb = grid_xgb.best_estimator_

    y_pred = best_xgb.predict(X_test)
    y_proba = best_xgb.predict_proba(X_test)[:, 1]


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_xgb.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "XGBoost",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_xgb.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.predict_proba(X_test)[:, 1]

            tuning_results.append({
                "model": "XGBoost",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Colunas utilizadas para o cenario 'sem_submodalidade': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários mínimos', '

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,XGBoost,sem_submodalidade,True,tuning_cv,0.919212,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:00.160313
1,XGBoost,sem_submodalidade,True,test,0.919416,0.850279,0.833692,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:06.434732
2,XGBoost,sem_submodalidade,False,tuning_cv,0.926068,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:06.435571
3,XGBoost,sem_submodalidade,False,test,0.926908,0.862829,0.845775,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:09.789878



🔎 Scenario: submodalidade_agrupada
Colunas utilizadas para o cenario 'submodalidade_agrupada': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários m

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,XGBoost,sem_submodalidade,True,tuning_cv,0.919212,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:00.160313
1,XGBoost,sem_submodalidade,True,test,0.919416,0.850279,0.833692,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:06.434732
2,XGBoost,sem_submodalidade,False,tuning_cv,0.926068,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:06.435571
3,XGBoost,sem_submodalidade,False,test,0.926908,0.862829,0.845775,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:09.789878
4,XGBoost,submodalidade_agrupada,True,tuning_cv,0.939389,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:32:39.024689
5,XGBoost,submodalidade_agrupada,True,test,0.939418,0.874153,0.858952,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:32:47.064786
6,XGBoost,submodalidade_agrupada,False,tuning_cv,0.942814,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:32:47.065498
7,XGBoost,submodalidade_agrupada,False,test,0.943110,0.880017,0.864191,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:32:50.143136



🔎 Scenario: submodalidade_engineered
Colunas utilizadas para o cenario 'submodalidade_engineered': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salári

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,XGBoost,sem_submodalidade,True,tuning_cv,0.919212,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:00.160313
1,XGBoost,sem_submodalidade,True,test,0.919416,0.850279,0.833692,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:06.434732
2,XGBoost,sem_submodalidade,False,tuning_cv,0.926068,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:06.435571
3,XGBoost,sem_submodalidade,False,test,0.926908,0.862829,0.845775,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:09.789878
4,XGBoost,submodalidade_agrupada,True,tuning_cv,0.939389,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:32:39.024689
5,XGBoost,submodalidade_agrupada,True,test,0.939418,0.874153,0.858952,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:32:47.064786
6,XGBoost,submodalidade_agrupada,False,tuning_cv,0.942814,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:32:47.065498
7,XGBoost,submodalidade_agrupada,False,test,0.943110,0.880017,0.864191,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:32:50.143136
8,XGBoost,submodalidade_engineered,True,tuning_cv,0.933313,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:43:58.613769
9,XGBoost,submodalidade_engineered,True,test,0.933226,0.867419,0.851652,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:44:05.077899


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,XGBoost,sem_submodalidade,True,tuning_cv,0.919212,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:00.160313
1,XGBoost,sem_submodalidade,True,test,0.919416,0.850279,0.833692,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:20:06.434732
2,XGBoost,sem_submodalidade,False,tuning_cv,0.926068,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:06.435571
3,XGBoost,sem_submodalidade,False,test,0.926908,0.862829,0.845775,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:20:09.789878
4,XGBoost,submodalidade_agrupada,True,tuning_cv,0.939389,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:32:39.024689
5,XGBoost,submodalidade_agrupada,True,test,0.939418,0.874153,0.858952,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:32:47.064786
6,XGBoost,submodalidade_agrupada,False,tuning_cv,0.942814,NaN,NaN,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:32:47.065498
7,XGBoost,submodalidade_agrupada,False,test,0.943110,0.880017,0.864191,"{'smote': 'passthrough', 'xgb__colsample_bytre...",2026-06-06 19:32:50.143136
8,XGBoost,submodalidade_engineered,True,tuning_cv,0.933313,NaN,NaN,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:43:58.613769
9,XGBoost,submodalidade_engineered,True,test,0.933226,0.867419,0.851652,"{'smote': SMOTE(random_state=42), 'xgb__colsam...",2026-06-06 19:44:05.077899
